In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [ ]:
file_path = os.path.join("..", "data", "processed", "removed_high_corr.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
data = pd.read_csv(file_path, index_col=0)
data

In [ ]:
data.columns

# 1. Plotting RFECV Results (Optimal Number of Features)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

# Set random seed globally (optional but can help)
import numpy as np
np.random.seed(42)

# Define features (X) and target (y)
X = data.drop(columns=['Exit_Status'])  # Drop target column
y = data['Exit_Status']  # Target variable

# Base Random Forest model
base_rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=1,                # Set to 1 for strict determinism
    max_depth=10,
    min_samples_split=5,
    max_features='sqrt'
)

# RFECV with StratifiedKFold using a fixed random_state
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rfecv = RFECV(
    estimator=base_rf_model,
    step=2,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=1  # Also set to 1 here
)

# Fit RFECV
rfecv.fit(X, y)

# Plot results
plt.figure(figsize=(8, 6))
plt.plot(
    range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    rfecv.cv_results_['mean_test_score']
)
plt.xlabel("Number of Features")
plt.ylabel("Cross-Validation Score (F1 Macro)")
plt.title("RFECV - Feature Selection Performance")
plt.grid(True)
plt.tight_layout()
plt.show()

# Selected features
X_selected = X.loc[:, rfecv.support_]
print(f"Optimal number of features: {rfecv.n_features_}")
print("Selected Features:")
print(X_selected.columns)

# Re-train model only on selected features
final_rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=1,
    max_depth=10,
    min_samples_split=5,
    max_features='sqrt'
)
final_rf.fit(X_selected, y)

# Feature importances
importances = pd.Series(final_rf.feature_importances_, index=X_selected.columns)
importances = importances.sort_values(ascending=False)

print("Top 10 Most Important Features:")
print(importances.head(10))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Get selected features based on RFE support mask
selected_features = X.columns[rfecv.support_]  # Only keep selected features

# Extract feature importances for the selected features
feature_importances = pd.Series(rfecv.estimator_.feature_importances_, index=selected_features)

# Get the top 10 most important features
top_features = feature_importances.nlargest(10)

# Reverse the gradient so the top bar is darkest
num_features = len(top_features)
colors = [(0, 0.5, 0, alpha) for alpha in np.linspace(0.3, 1, num_features)]  # Darker green at the top

# Plot feature importance with reversed green gradient
plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.xlabel("Feature Importance Score", fontsize=12, fontweight="bold")
plt.ylabel("Feature Name", fontsize=12, fontweight="bold")
plt.title("Top 10 Most Important Features", fontsize=14, fontweight="bold", pad=10)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# Create a new DataFrame with the selected features + target variable
data_subset = data[selected_features.tolist() + ['Exit_Status']]

# Display the data frame with the selected features + target variable using pandas
print("Subset Data Ready for Prediction:")
data_subset


# Heuristic Approach:

80/20 Rule (Pareto Principle): A common rule of thumb is to retain around 20% of the original features, 
especially in high-dimensional datasets. In your case, with 142 features, this would mean retaining roughly 28 features.
Lets return 28 features out of the 49 features above. 

In [ ]:
'''
# Get selected features based on RFE support mask
selected_features = X.columns[rfecv.support_]  # Only keep selected features

# Extract feature importances for the selected features
feature_importances = pd.Series(rfecv.estimator_.feature_importances_, index=selected_features)

# Get the top 28 most important features
top_28_features = feature_importances.nlargest(28)

# Create a new DataFrame with the top 28 most important features + target variable
data_subset = data[top_28_features.index.tolist() + ['Exit_Status']]

# Display the top 10 most important features
top_10_features = top_28_features.nlargest(10)

# Reverse the gradient so the top bar is darkest
num_features = len(top_10_features)
colors = [(0, 0.5, 0, alpha) for alpha in np.linspace(0.3, 1, num_features)]  # Darker green at the top

# Plot feature importance with reversed green gradient
plt.figure(figsize=(10, 6))
top_10_features.sort_values().plot(kind='barh', color=colors, edgecolor='black')

plt.xlabel("Feature Importance Score", fontsize=12, fontweight="bold")
plt.ylabel("Feature Name", fontsize=12, fontweight="bold")
plt.title("Top 10 Most Important Features", fontsize=14, fontweight="bold", pad=10)
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# Display the first few rows of the top 28 features with the target variable
print("Subset Data Ready for Prediction:")
data_subset
'''

# Train Predictive Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# Define X (all features except the target variable)
X = data_subset.drop(columns=['Exit_Status'])  # Drop target column
y = data_subset['Exit_Status']  # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Print shape of training and testing sets
print(f"Training data shape: {X_train.shape}, {y_train.shape}")
print(f"Testing data shape: {X_test.shape}, {y_test.shape}")


# More Performance Matrices to Evaluate models

1. Precision, Recall, and F1-Score
✅ Precision (Positive Predictive Value): Measures how many predicted positives are actually correct.
✅ Recall (Sensitivity): Measures how many actual positives were correctly predicted.
✅ F1-Score: Harmonic mean of Precision and Recall, useful for imbalanced datasets.

2. Confusion Matrix
✅ Gives a breakdown of True Positives, True Negatives, False Positives, and False Negatives.
✅ Useful for understanding misclassifications.

3. ROC-AUC Score (for Multi-Class Classification)
✅ ROC Curve: Plots True Positive Rate (TPR) vs. False Positive Rate (FPR).
✅ AUC (Area Under Curve): Measures overall model performance.

4. Matthews Correlation Coefficient (MCC)
✅ Works well even for imbalanced datasets.
✅ Returns a value between -1 (bad) and +1 (perfect).

5. Log Loss (Categorical Cross-Entropy)
✅ Penalizes incorrect confident predictions more heavily.
✅ Lower log loss means better calibration.

Which Metric Should You Use?
Scenario	Best Metrics
Balanced Classes	Accuracy, F1-Score
Imbalanced Classes	Precision, Recall, F1-Score, MCC
Multi-Class Problems	ROC-AUC, Log Loss
Business-Critical Decisions (e.g., Fraud Detection)	Precision, Recall (depending on false positive/false negative impact)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import label_binarize

# Standardization
scaler = StandardScaler()

# Feature Selection using L1 Regularization
feature_selector = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', random_state=42))

# Optimized Logistic Regression with Hyperparameter Tuning
lr_model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='saga', random_state=42)

# Pipeline for Scaling, Feature Selection, and Model Training
pipeline = Pipeline([
    ('scaler', scaler),
    ('feature_selection', feature_selector),
    ('log_reg', lr_model)
])

# Hyperparameter tuning
param_grid = {
    'log_reg__C': [0.001, 0.01, 0.1, 1, 10]  # Regularization strength
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
lr_preds = best_model.predict(X_test)
lr_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC

# Compute Metrics
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_f1_score = f1_score(y_test, lr_preds, average='weighted')  # Weighted F1-score for multi-class
lr_roc_auc = roc_auc_score(y_test, lr_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Logistic Regression Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {lr_accuracy:.4f}")
print(f"F1-Score: {lr_f1_score:.4f}")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], lr_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], lr_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Define the pipeline with scaling
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Feature scaling
    ('svm', SGDClassifier(loss='log_loss', random_state=42))  # Change 'hinge' to 'log_loss'
])

# Hyperparameter grid
param_grid = {
    'svm__alpha': [0.0001, 0.001, 0.01, 0.1],  # Regularization strength
    'svm__max_iter': [1000, 2000, 3000],  # Number of iterations
    'svm__penalty': ['l2', 'l1', 'elasticnet'],  # Regularization type
    'svm__early_stopping': [True]  # Enables early stopping
}

# Grid search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
svm_preds = best_model.predict(X_test)
svm_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC and AUC

# Compute Metrics
svm_accuracy = accuracy_score(y_test, svm_preds)
svm_f1_score = f1_score(y_test, svm_preds, average='weighted')  # Weighted F1-score for multi-class
svm_roc_auc = roc_auc_score(y_test, svm_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Support Vector Machine (SGDClassifier) Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Accuracy: {svm_accuracy:.4f}")
print(f"F1-Score: {svm_f1_score:.4f}")
print(f"ROC-AUC Score: {svm_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], svm_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], svm_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification (SVM)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Define Random Forest model
rf_model = RandomForestClassifier(
    oob_score=True,  # Use out-of-bag samples for validation
    n_jobs=-1,       # Enable parallel processing
    random_state=42
)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # More trees for better performance
    'max_depth': [10, 20, None],  # Try deeper trees for complex patterns
    'min_samples_split': [5, 10, 20],  # Prevent overfitting
    'min_samples_leaf': [1, 5, 10],  # Control leaf size
    'max_features': ['sqrt', 'log2', None],  # Optimize feature selection per tree
    'class_weight': ['balanced', None]  # Handle class imbalance
}

# Grid search with cross-validation
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
rf_preds = best_model.predict(X_test)
rf_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC and AUC

# Compute Metrics
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_f1_score = f1_score(y_test, rf_preds, average='weighted')  # Weighted F1-score for multi-class
rf_roc_auc = roc_auc_score(y_test, rf_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Random Forest Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"OOB Score: {best_model.oob_score_:.4f}")  # Out-of-bag validation score
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-Score: {rf_f1_score:.4f}")
print(f"ROC-AUC Score: {rf_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], rf_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification (Random Forest)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()


# Calculations of risk scores

We see that random forest has the highest accuracy.

Determine the optimal threshold for the risk score.

Model Evaluation Metrics:
Out-of-Bag (OOB) Score: 0.6818 — This is the average error rate when using bootstrap samples to build trees.

Accuracy: 0.6721 — This is the overall accuracy of the model, meaning around 67% of the predictions were correct.

F1-Score: 0.6290 — This is the harmonic mean of precision and recall. It's a good measure of model performance, especially when you care about both false positives and false negatives.

Next Steps for Threshold Selection:
To find the optimal threshold, we can:

Plot the ROC curve and calculate AUC.

Plot the Precision-Recall curve and find the threshold that maximizes the F1-score.

Use Youden’s Index to select the optimal threshold.

Consider the business implications of false positives and false negatives, adjusting the threshold accordingly.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.preprocessing import label_binarize
import numpy as np

# Define Random Forest model
rf_model = RandomForestClassifier(
    oob_score=True,  # Use out-of-bag samples for validation
    n_jobs=-1,       # Enable parallel processing
    random_state=42
)

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],  # More trees for better performance
    'max_depth': [10, 20, None],  # Try deeper trees for complex patterns
    'min_samples_split': [5, 10, 20],  # Prevent overfitting
    'min_samples_leaf': [1, 5, 10],  # Control leaf size
    'max_features': ['sqrt', 'log2', None],  # Optimize feature selection per tree
    'class_weight': ['balanced', None]  # Handle class imbalance
}

# Grid search with cross-validation
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Model
best_model = grid_search.best_estimator_

# Predictions
rf_preds = best_model.predict(X_test)
rf_probs = best_model.predict_proba(X_test)  # Get predicted probabilities for ROC and AUC

# Compute Metrics
rf_accuracy = accuracy_score(y_test, rf_preds)
rf_f1_score = f1_score(y_test, rf_preds, average='weighted')  # Weighted F1-score for multi-class
rf_roc_auc = roc_auc_score(y_test, rf_probs, multi_class='ovr')  # ROC-AUC for multi-class

# Display performance metrics
print("Optimized Random Forest Model:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"OOB Score: {best_model.oob_score_:.4f}")  # Out-of-bag validation score
print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-Score: {rf_f1_score:.4f}")
print(f"ROC-AUC Score: {rf_roc_auc:.4f}")

# Plot ROC Curves for each class
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])  # Binarize true labels for multi-class ROC
n_classes = y_test_bin.shape[1]

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], rf_probs[:, i])
    plt.plot(fpr, tpr, label=f'Class {i} (AUC = {roc_auc_score(y_test_bin[:, i], rf_probs[:, i]):.2f})')

# Plot diagonal line (random classifier)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Classifier')

# Customize plot
plt.title('ROC Curves for Multi-Class Classification (Random Forest)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# Step 1: Determine Optimal Threshold Using ROC Curve (Youden’s Index)
# Calculate Youden's Index for each class and find the best threshold for each class
youden_thresholds = []
for i in range(n_classes):
    fpr, tpr, thresholds_roc = roc_curve(y_test_bin[:, i], rf_probs[:, i])
    youdens_index = tpr - fpr
    best_threshold_youden = thresholds_roc[np.argmax(youdens_index)]
    youden_thresholds.append(best_threshold_youden)
    print(f"Best threshold for class {i} based on Youden's Index: {best_threshold_youden:.2f}")

# Step 2: Plot Precision-Recall Curve and Maximize F1-Score
# Precision-Recall curve for each class
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    precision, recall, thresholds_pr = precision_recall_curve(y_test_bin[:, i], rf_probs[:, i])
    f1_scores = 2 * (precision * recall) / (precision + recall)
    best_threshold_f1 = thresholds_pr[np.argmax(f1_scores)]
    print(f"Best threshold for class {i} based on F1 score: {best_threshold_f1:.2f}")

    # Plot PR curve
    plt.plot(recall, precision, label=f'Class {i} (F1 = {np.max(f1_scores):.2f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves for Multi-Class Classification')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()


Interpreting the Results:

Youden’s Index Thresholds:

Class 0 (graduating): 0.46

Class 1 (switching): 0.35

Class 2 (dropping): 0.31

Youden’s Index maximizes the difference between True Positive Rate (TPR) and False Positive Rate (FPR). These thresholds ensure a balance between correctly identifying high-risk students and minimizing false positives.

F1-Score Thresholds:

Class 0 (graduating): 0.46

Class 1 (switching): 0.35

Class 2 (dropping): 0.26

F1-score optimizes the balance between precision and recall. The thresholds based on the F1 score ensure that you maximize the model's ability to both detect high-risk students and avoid incorrect interventions.

Choosing the Optimal Threshold for Intervention:
Threshold for Class 0 (graduating):

Both Youden’s Index and F1 Score suggest a threshold of around 0.46.

Since graduating students are typically low-risk, this threshold will help flag only the most risky students.

In this case, you might not need much intervention for this class, but you’ll still track those that are at high risk of leaving successfully.

Threshold for Class 1 (switching):

Youden’s Index suggests a lower threshold of 0.20, which identifies a broader group of students at risk of switching internally.

F1-Score suggests a slightly higher threshold of 0.37, which may better balance false positives and false negatives.

Depending on your tolerance for false positives, you can choose between 0.20 (more inclusive) and 0.37 (more conservative). In general, a threshold of 0.37 is a reasonable choice as it captures more high-risk students without too many low-risk students being flagged.

Threshold for Class 2 (dropping):

Youden’s Index suggests 0.31, and F1-Score suggests 0.30.

These thresholds capture students who are at risk of leaving unsuccessfully, a critical group for intervention.

Given these thresholds are very close, you can select 0.31 or 0.30 based on your desired sensitivity and specificity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the thresholds based on Youden's Index or F1 score as per the results you provided
thresholds = {
    0: 0.45,  # Threshold for class 0 (graduating)
    1: 0.37,  # Threshold for class 1 (switching)
    2: 0.30   # Threshold for class 2 (dropping)
}

# Use the predicted probabilities (rf_probs) that you already have
# Assuming rf_probs contains the probabilities for each class (0, 1, 2)
graduating_risk = rf_probs[:, 0]  # Class 0 (graduating)
switching_risk = rf_probs[:, 1]  # Class 1 (switching)
dropping_risk = rf_probs[:, 2]  # Class 2 (dropping)

# Apply thresholds to calculate risk for each category
# Classifying based on thresholds
reliable = graduating_risk <= thresholds[0]
high_risk_switching = switching_risk >= thresholds[1]
high_risk_leaving = dropping_risk >= thresholds[2]

# Print the results
print("Reliable (Class 0 - Leaving Successfully):")
print(reliable[:10])  # Display first 10 values as an example

print("High Risk for Switching Internally (Class 1):")
print(high_risk_switching[:10])  # Display first 10 values as an example

print("High Risk for Leaving Unsuccessfully (Class 2):")
print(high_risk_leaving[:10])  # Display first 10 values as an example

# Optionally, calculate the proportion of each category being classified as reliable/high risk
print("\nProportion of students classified as reliable (Class 0):", np.mean(reliable))
print("Proportion of students classified as high risk for switching internally (Class 1):", np.mean(high_risk_switching))
print("Proportion of students classified as high risk for leaving unsuccessfully (Class 2):", np.mean(high_risk_leaving))

# Plot histograms for the risk scores and the threshold lines
plt.figure(figsize=(12, 8))

# Plot for "Leaving Successfully" risk scores (Class 0)
plt.subplot(131)
plt.hist(graduating_risk, bins=20, color='blue', alpha=0.7)
plt.axvline(x=thresholds[0], color='red', linestyle='--')
plt.title('Graduating (Class 0)')
plt.xlabel('Risk Score')
plt.ylabel('Frequency')

# Plot for "Switching Internally" risk scores (Class 1)
plt.subplot(132)
plt.hist(switching_risk, bins=20, color='orange', alpha=0.7)
plt.axvline(x=thresholds[1], color='red', linestyle='--')
plt.title('Switching (Class 1)')
plt.xlabel('Risk Score')
plt.ylabel('Frequency')

# Plot for "Leaving Unsuccessfully" risk scores (Class 2)
plt.subplot(133)
plt.hist(dropping_risk, bins=20, color='green', alpha=0.7)
plt.axvline(x=thresholds[2], color='red', linestyle='--')
plt.title('Dropping (Class 2)')
plt.xlabel('Risk Score')
plt.ylabel('Frequency')

# Add a single legend for all subplots
plt.figlegend(['Risk score threshold'], 
              loc='lower right', ncol=1, frameon=False, bbox_to_anchor=(0.5, -0.05))

# Adjust layout to make space for the legend
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space at the bottom for the legend

plt.show()
